# 研究模板：K 线 / 成交 / 回测结果

前置：先运行 `quant research export ...` 生成数据目录（见 `notebooks/README.md`）。

原则：本 notebook **只读导出目录**，不连 SQLite / Redis / OKX。

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

DATA_DIR = Path(os.environ.get("QUANT_RESEARCH_DIR", "research/BTC-USDT-1H"))
BARS = DATA_DIR / "bars.parquet"
TRADES = DATA_DIR / "trades.parquet"
BACKTEST = DATA_DIR / "backtest.json"

if not BARS.is_file():
    raise SystemExit(
        f"未找到 {BARS}：先运行 `quant research export --inst BTC-USDT "
        "--bar 1H --start 2026-09-01 --end 2026-09-10`"
    )

print("数据目录：", DATA_DIR.resolve())

In [ ]:
bars = pd.read_parquet(BARS)
for col in ("open", "high", "low", "close", "vol", "vol_quote"):
    if col in bars.columns:
        bars[col] = pd.to_numeric(bars[col], errors="coerce")

print("K线行数：", len(bars))
print("时间范围：", bars["ts"].min(), "->", bars["ts"].max())
print("缺失值：")
print(bars[["open", "high", "low", "close", "vol"]].isna().sum())
bars.head()

In [ ]:
if TRADES.is_file():
    trades = pd.read_parquet(TRADES)
    for col in ("qty", "price", "fee"):
        if col in trades.columns:
            trades[col] = pd.to_numeric(trades[col], errors="coerce")
    print("成交笔数：", len(trades))
    if len(trades):
        print(
            trades.groupby("side").agg(
                count=("qty", "size"), qty=("qty", "sum"), fee=("fee", "sum")
            )
        )
        print("名义金额合计：", float((trades["qty"] * trades["price"]).sum()))
    trades.head()
else:
    print("没有 trades.parquet（导出时加 --run-id 可带上回测成交）")

In [ ]:
if BACKTEST.is_file():
    payload = json.loads(BACKTEST.read_text(encoding="utf-8"))
    run = payload.get("run") or {}
    print("run：", run.get("run_id"), "·", run.get("strategy_id"))
    print("区间：", run.get("start_ts"), "->", run.get("end_ts"))
    print("参数：", run.get("params"))
    for key, value in sorted((payload.get("metrics") or {}).items()):
        if key != "run_id":
            print(f"{key}: {value}")
else:
    print("没有 backtest.json（导出时加 --run-id）")

## 下一步

- 改参数重跑 → `quant research export ... --run-id <new>` → 对比 `metrics`
- 需要画图：`uv pip install matplotlib`，然后
  `bars.set_index("ts")["close"].plot()`
- 结论里记录 `manifest.json` 的 `exported_at` / `run_id`，保证可复现